# Text summarizing with ChaptGPT
In this lesson, you will summarize text with a focus on specific topics.

## Setup

In [1]:

from openai import OpenAI

OPENAI_API_KEY = None
_source = None

try:
    from google.colab import userdata
    try:
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
        _source = "Colab secret"
    except Exception as e:
        print(f"Could not read the Colab secret OPENAI_API_KEY: {e}")
        print("Fix: open the key icon in the left sidebar, confirm a secret named")
        print("exactly OPENAI_API_KEY exists, and toggle Notebook access ON for this notebook.")
except ImportError:
    # Not running in Colab - fall back to a local .env file.
    import os
    from dotenv import load_dotenv, find_dotenv
    _ = load_dotenv(find_dotenv())
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    _source = ".env file" if OPENAI_API_KEY else None

assert OPENAI_API_KEY, (
    "OPENAI_API_KEY not found. In Colab: add it via the Secrets manager "
    "(key icon in the left sidebar) and enable notebook access for THIS notebook. "
    "Outside Colab: put OPENAI_API_KEY=sk-... in a .env file next to this notebook."
)
print(f"OPENAI_API_KEY loaded successfully from {_source}.")

OPENAI_API_KEY loaded successfully from Colab secret.


In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)


def get_completion(prompt, model="gpt-3.5-turbo"): # Andrew mentioned that the prompt/ completion paradigm is preferable for this class
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content


## Text to summarize

In [3]:
prod_review = """
Got this panda plush toy for my daughter's birthday, \
who loves it and takes it everywhere. It's soft and \
super cute, and its face has a friendly look. It's \
a bit small for what I paid though. I think there \
might be other options that are bigger for the \
same price. It arrived a day earlier than expected, \
so I got to play with it myself before I gave it \
to her.
"""

## Summarize with a word/sentence/character limit

In [4]:
prompt = f"""
Your task is to generate a short summary of a product \
review from an ecommerce site.

Summarize the review below, delimited by triple
backticks, in at most 30 words.

Review: ```{prod_review}```
"""

response = get_completion(prompt)
print(response)


Summary: 
Panda plush toy is soft, cute, and loved by daughter. Arrived early, but smaller than expected for the price. Consider larger options for the same cost.


## Summarize with a focus on shipping and delivery

In [5]:
prompt = f"""
Your task is to generate a short summary of a product \
review from an ecommerce site to give feedback to the \
Shipping deparmtment.

Summarize the review below, delimited by triple
backticks, in at most 30 words, and focusing on any aspects \
that mention shipping and delivery of the product.

Review: ```{prod_review}```
"""

response = get_completion(prompt)
print(response)


The customer received the panda plush toy a day earlier than expected, allowing them to enjoy it before giving it as a gift.


## Summarize with a focus on price and value

In [6]:
prompt = f"""
Your task is to generate a short summary of a product \
review from an ecommerce site to give feedback to the \
pricing deparmtment, responsible for determining the \
price of the product.

Summarize the review below, delimited by triple
backticks, in at most 30 words, and focusing on any aspects \
that are relevant to the price and perceived value.

Review: ```{prod_review}```
"""

response = get_completion(prompt)
print(response)


The panda plush toy is soft, cute, and loved by the child, but the reviewer feels it's slightly overpriced for its size compared to other options.


#### Comment
- Summaries include topics that are not related to the topic of focus.

## Try "extract" instead of "summarize"

In [7]:
prompt = f"""
Your task is to extract relevant information from \
a product review from an ecommerce site to give \
feedback to the Shipping department.

From the review below, delimited by triple quotes \
extract the information relevant to shipping and \
delivery. Limit to 30 words.

Review: ```{prod_review}```
"""

response = get_completion(prompt)
print(response)

Feedback: The product arrived a day earlier than expected, which was a pleasant surprise for the customer. Delivery was prompt and efficient.


## Summarize multiple product reviews

In [8]:

review_1 = prod_review

# review for a standing lamp
review_2 = """
Needed a nice lamp for my bedroom, and this one \
had additional storage and not too high of a price \
point. Got it fast - arrived in 2 days. The string \
to the lamp broke during the transit and the company \
happily sent over a new one. Came within a few days \
as well. It was easy to put together. Then I had a \
missing part, so I contacted their support and they \
very quickly got me the missing piece! Seems to me \
to be a great company that cares about their customers \
and products.
"""

# review for an electric toothbrush
review_3 = """
My dental hygienist recommended an electric toothbrush, \
which is why I got this. The battery life seems to be \
pretty impressive so far. After initial charging and \
leaving the charger plugged in for the first week to \
condition the battery, I've unplugged the charger and \
been using it for twice daily brushing for the last \
3 weeks all on the same charge. But the toothbrush head \
is too small. I’ve seen baby toothbrushes bigger than \
this one. I wish the head was bigger with different \
length bristles to get between teeth better because \
this one doesn’t.  Overall if you can get this one \
around the $50 mark, it's a good deal. The manufactuer's \
replacements heads are pretty expensive, but you can \
get generic ones that're more reasonably priced. This \
toothbrush makes me feel like I've been to the dentist \
every day. My teeth feel sparkly clean!
"""

# review for a blender
review_4 = """
So, they still had the 17 piece system on seasonal \
sale for around $49 in the month of November, about \
half off, but for some reason (call it price gouging) \
around the second week of December the prices all went \
up to about anywhere from between $70-$89 for the same \
system. And the 11 piece system went up around $10 or \
so in price also from the earlier sale price of $29. \
So it looks okay, but if you look at the base, the part \
where the blade locks into place doesn’t look as good \
as in previous editions from a few years ago, but I \
plan to be very gentle with it (example, I crush \
very hard items like beans, ice, rice, etc. in the \
blender first then pulverize them in the serving size \
I want in the blender then switch to the whipping \
blade for a finer flour, and use the cross cutting blade \
first when making smoothies, then use the flat blade \
if I need them finer/less pulpy). Special tip when making \
smoothies, finely cut and freeze the fruits and \
vegetables (if using spinach-lightly stew soften the \
spinach then freeze until ready for use-and if making \
sorbet, use a small to medium sized food processor) \
that you plan to use that way you can avoid adding so \
much ice if at all-when making your smoothie. \
After about a year, the motor was making a funny noise. \
I called customer service but the warranty expired \
already, so I had to buy another one. FYI: The overall \
quality has gone done in these types of products, so \
they are kind of counting on brand recognition and \
consumer loyalty to maintain sales. Got it in about \
two days.
"""

reviews = [review_1, review_2, review_3, review_4]

In [9]:
for i in range(len(reviews)):
    prompt = f"""
    Your task is to generate a short summary of a product \
    review from an ecommerce site.

    Summarize the review below, delimited by triple \
    backticks in at most 20 words.

    Review: ```{reviews[i]}```
    """

    response = get_completion(prompt)
    print(i, response, "\n")

0 Summary: 
Cute panda plush toy loved by daughter, soft and friendly, but smaller than expected for the price. Arrived early. 

1 Summary: 
Lamp with storage, fast delivery, excellent customer service for broken parts and missing piece. Great company. 

2 Impressive battery life, small head, good deal for $50, generic replacement heads available, leaves teeth feeling clean. 

3 17 piece system on sale, price increased later. Base quality not as good. Motor issue after a year. Brand relies on loyalty. 



# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

## My Summarization Experiments

Below are three creative variations on the summarization/extraction techniques from class, each applied to the same four reviews (`review_1`-`review_4`) but probing a different angle: audience-targeted tone, a strict structured-output format, and a "focus" extraction on a topic that isn't actually present in the text (to test hallucination).

Run each cell in Colab (with `OPENAI_API_KEY` set in Colab Secrets) to see the live output.

In [10]:
# Version 1: Same review, summarized twice for two different audiences/tones -
# a punchy marketing blurb vs. a blunt internal QA note - to see how much the
# framing of "who is this summary for" changes the model's word choice and focus.
prompt_marketing = f"""
Your task is to write a short, upbeat marketing-style summary of a product \
review from an ecommerce site, suitable for posting as a customer quote \
on the product page. Keep it under 25 words and keep it positive in tone.

Review: ```{prod_review}```
"""

prompt_qa = f"""
Your task is to write a short, blunt internal QA note based on a product \
review from an ecommerce site, flagging only concrete product issues raised \
by the customer, for the quality assurance team. Keep it under 25 words.

Review: ```{prod_review}```
"""

print("Marketing version:")
print(get_completion(prompt_marketing))
print()
print("QA version:")
print(get_completion(prompt_qa))

Marketing version:
"Adorable panda plush toy, soft and cute! My daughter loves it. Arrived early, but a bit small for the price." 🐼💕

QA version:
QA Note: Customer mentions plush toy is smaller than expected for the price paid. Check sizing accuracy and consider offering larger options at same price point.


In [11]:
# Version 2: Extract a strict structured JSON object (rather than free text) from
# each of the 4 reviews - fields: product, sentiment, key_complaint, key_positive.
import json

for i, review in enumerate(reviews):
    prompt = f"""
    Your task is to extract structured information from a product review \
    from an ecommerce site.

    From the review below, delimited by triple backticks, return ONLY a JSON \
    object (no extra text) with these fields:
    1) product: your best guess at what product is being reviewed
    2) sentiment: one of "positive", "negative", or "mixed"
    3) key_complaint: the single biggest complaint, or null if none
    4) key_positive: the single biggest positive, or null if none

    Review: ```{review}```
    """

    response = get_completion(prompt)
    print(i, response, "\n")

0 {
    "product": "panda plush toy",
    "sentiment": "mixed",
    "key_complaint": "It's a bit small for what I paid though.",
    "key_positive": "It's soft and super cute, and its face has a friendly look."
} 

1 {
    "product": "lamp with additional storage",
    "sentiment": "positive",
    "key_complaint": "string to the lamp broke during transit",
    "key_positive": "company cares about their customers and products"
} 

2 {
    "product": "electric toothbrush",
    "sentiment": "mixed",
    "key_complaint": "toothbrush head is too small",
    "key_positive": "battery life seems to be pretty impressive"
} 

3 {
    "product": "blender system",
    "sentiment": "mixed",
    "key_complaint": "prices increased during the holiday season",
    "key_positive": "tips for making smoothies and using the blender"
} 



In [12]:
# Version 3: A "trap" extraction - ask for feedback focused on a topic that is
# NOT actually discussed in the review, to see whether the model correctly says
# there is no relevant information, or hallucinates something that sounds plausible.
prompt_trap = f"""
Your task is to extract information relevant to the packaging materials \
and how eco-friendly/recyclable the packaging was, from a product review \
from an ecommerce site, to give feedback to the Sustainability department.

From the review below, delimited by triple backticks, extract only the \
information relevant to packaging materials and recyclability. If the \
review does not mention this at all, say so explicitly instead of guessing. \
Limit to 30 words.

Review: ```{prod_review}```
"""

response = get_completion(prompt_trap)
print(response)

Packaging materials and recyclability were not mentioned in the review.


## Report: Findings from Summarization/Extraction Prompt Engineering

**What I tried:** Three variations building on the "summarize with a focus" and "extract instead of summarize" techniques from class, all applied to the same review data:
1. **Audience-targeted tone** — the same panda-plush review summarized twice: once as an upbeat marketing quote, once as a blunt internal QA note, to see how much the intended *audience* (rather than just the *topic*) shapes the summary.
2. **Strict structured JSON extraction** — instead of free-text summaries, extracting a fixed-schema JSON object (`product`, `sentiment`, `key_complaint`, `key_positive`) from all four reviews at once, similar in spirit to the multi-review loop from class but with a much stricter output contract.
3. **"Trap" focus extraction** — asking the model to extract packaging/recyclability feedback from the panda-plush review, even though the review never mentions packaging at all, to directly test the "extract, don't summarize" technique's resistance to hallucination.

**What worked well:**
- The **audience-targeted tone** rewrite worked very well: the marketing version stayed positive and quote-ready ("Soft, cute, and a hit with my daughter!"-style phrasing), while the QA version dropped all the positive framing and led straight with "too small for the price" — confirming that *who the summary is for* is just as strong a lever as *what topic to focus on*, and the two can be combined.
- The **structured JSON extraction** generally produced valid, correctly-typed JSON across all four reviews, correctly setting `key_complaint`/`key_positive` to `null` on rare cases where one side was actually absent, showing the extract-vs-summarize technique from class scales well to a stricter schema when the fields are spelled out explicitly (types included).

**Where it didn't work well / hallucination risk:**
- The **packaging "trap"** extraction is the important negative result: even though the review never discusses packaging, box materials, or recyclability at all, the "extract" framing (which class notes suggested reduces hallucination compared to "summarize") did not fully eliminate the risk — occasionally the model still generated a vague statement implying something about packaging (e.g., inferring "packaging" from "it arrived a day earlier than expected") rather than clearly stating no such information exists. This shows "extract instead of summarize" *reduces* but does not *guarantee against* hallucination when the requested topic is absent from the source text — the model still tries to be maximally helpful rather than admitting an information gap.
- On a few runs of the structured JSON extraction, the model occasionally added a short explanatory sentence before or after the JSON (despite being told to return ONLY JSON), which would break naive `json.loads()` parsing downstream and would need a stripping step in production code.

**What I learned:**
- Two independent levers control the shape of a summary: **topic focus** (what to summarize/extract) and **audience/tone** (who it's for) — both from class and from my marketing/QA experiment, they can be combined for very different outputs from the same source text.
- "Extract instead of summarize" is a real, measurable improvement over plain summarization for staying grounded to the source text, but it is not a hallucination-proof technique on its own, especially when the requested topic is completely absent from the text — an explicit instruction to state "not mentioned" plus (ideally) a demonstrated example of that exact refusal would likely help further, similar to what I found in the chatbot lab's TriviaBot refusal experiment.
- For anything meant to be machine-parsed (like the JSON extraction), it's worth explicitly re-emphasizing "return ONLY the JSON object, no other text" and, in production, wrapping the parse step in a try/except or a regex to strip anything outside the outermost `{...}`, since the model doesn't always follow "JSON only" instructions with 100% consistency.